In [ ]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.model_selection import GridSearchCV

: 

In [ ]:
# Loading the data 

df = pd.read_csv("processed_crop_dataset.csv")

In [ ]:
# Seperating the lable and featurs for before splitting 

X = df.drop("label", axis=1)
y = df["label"]

In [ ]:
# Splitting the dataset into 80% training data and 20% testing data.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
# Defining multiple classification algorithms for initial model comparison.
# StandardScaler is used through pipelines for models that are sensitive to feature scale.

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

In [ ]:
# Training each classification model on the training dataset.
# Predictions are generated on the test dataset and evaluated using [Accuracy, Macro Precision, Macro Recall, and Macro F1 Score].


results = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

results_df = pd.DataFrame(results)

# The models are then compared based on their F1 Score.
print(results_df.sort_values("F1 Score", ascending=False))

In [ ]:
# Performing 5-fold stratified cross-validation on the training data

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for name, model in models.items():
    print(f"Cross-validating {name}...")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "precision": "precision_macro",
            "recall": "recall_macro",
            "f1": "f1_macro"
        },
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "CV Accuracy Mean": scores["test_accuracy"].mean(),
        "CV Accuracy Std": scores["test_accuracy"].std(),
        "CV Precision Mean": scores["test_precision"].mean(),
        "CV Recall Mean": scores["test_recall"].mean(),
        "CV F1 Mean": scores["test_f1"].mean(),
        "CV F1 Std": scores["test_f1"].std()
    })

cv_results_df = pd.DataFrame(cv_results)

print(
    cv_results_df
    .sort_values("CV F1 Mean", ascending=False)
    .to_string(index=False)
)

In [ ]:
# Tuning the Random Forest hyperparameters using GridSearchCV.

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)

print("Best Parameters:")
print(rf_grid.best_params_)

print("\nBest CV Macro F1:")
print(rf_grid.best_score_)

In [ ]:
# Tuning the Extra Trees classifier using GridSearchCV.

et = ExtraTreesClassifier(
    random_state=42,
    n_jobs=-1
)

et_param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

et_grid = GridSearchCV(
    estimator=et,
    param_grid=et_param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

et_grid.fit(X_train, y_train)

print("Best Parameters:")
print(et_grid.best_params_)

print("\nBest CV Macro F1:")
print(et_grid.best_score_)

In [ ]:
# Selecting the best Random Forest model obtained from hyperparameter tuning.
# The tuned model is evaluated on the untouched test dataset

best_model = rf_grid.best_estimator_

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print("Final Test Results")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

In [ ]:
# Selecting the best Random Forest model obtained from hyperparameter tuning.
# The tuned model is evaluated on the untouched test dataset

In [ ]:
metrics = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1
}

plt.figure(figsize=(8, 5))
plt.bar(metrics.keys(), metrics.values(), color="#006572")


plt.ylim(0.98, 1.00)
plt.ylabel("Score")
plt.title("Final Random Forest Model Performance")

for i, value in enumerate(metrics.values()):
    plt.text(i, value + 0.0003, f"{value:.4f}", ha="center")

plt.tight_layout()
plt.show()

In [ ]:
plot_df = cv_results_df.sort_values("CV F1 Mean", ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(plot_df["Model"], plot_df["CV F1 Mean"], color = "maroon")

plt.ylim(0.80, 1.00)
plt.ylabel("Macro F1 Score")
plt.title("Model Comparison Using 5-Fold Cross-Validation")
plt.xticks(rotation=30, ha="right")

for i, value in enumerate(plot_df["CV F1 Mean"]):
    plt.text(i, value + 0.005, f"{value:.3f}", ha="center")

plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=best_model.classes_)

plt.figure(figsize=(18, 15))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=best_model.classes_,
    yticklabels=best_model.classes_
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Random Forest")
plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
#saving the best model

import joblib

joblib.dump(best_model, "crop_recommendation_model.pkl")
print("Model saved successfully.")